# Mid-Stream Scope Changes and Expectation Setting

**Phase 11** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-11/06-scope-changes-expectation-setting.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You are three weeks into a six-week engagement. The customer sends a Slack message: "Hey, while you are building the support ticket classifier, could you also make it handle billing questions? That is basically the same thing, right?"

This is one of the three most dangerous moments in an FDE engagement. Not because the request is unreasonable, but because how you respond in the next 60 seconds determines whether you finish on time and on budget, or whether you quietly agree to scope creep that stretches two weeks of work into five.

The failure mode is not malice. Customers genuinely do not know that "billing questions" means retraining on a different data distribution, adding a new intent ...

## The Concept

### The Three Scope Change Types

Every scope change request fits one of three categories. Classifying it correctly determines the right response.

```
CLARIFICATION
  Definition: Same goal, better definition.
  Example: "By 'support tickets' we mean only Tier 1, not billing or returns."
  Impact: Usually reduces scope. Sometimes changes priority.
  Response: Accept immediately, update scope doc, document what changed.

EXPANSION
  Definition: Same goal, more features or coverage.
  Example: "Can we also handle billing questions with the classifier?"
  Impact: Additional work. Days to weeks. M...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
ScopeChangeEvaluator: classify scope changes, estimate effort impact,
and draft a customer response.

Usage:
    python main.py --demo
    python main.py --context "..." --scope "..." --change "..."
"""

import argparse
import json
import os
import sys
from dataclasses import dataclass
from enum import Enum

import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-3-5-haiku-20241022"


class ChangeType(str, Enum):
    CLARIFICATION = "clarification"
    EXPANSION = "expansion"
    PIVOT = "pivot"


@dataclass
class ScopeChangeAssessment:
    change_type: ChangeType
    effort_delta_days: int
    timeline_risk: str       # "low", "medium", "high"
    rationale: str
    customer_response: str
    scope_doc_update: str


CLASSIFY_PROMPT = """You are an experienced AI engineering consultant assessing a scope change request.

Project context:
{context}

Original scope:
{original_scope}

Scope change request:
{change_request}

Classify this change and provide your assessment in this exact JSON format:
{{
  "change_type": "clarification" | "expansion" | "pivot",
  "effort_delta_days": <integer, 0 for clarification>,
  "timeline_risk": "low" | "medium" | "high",
  "rationale": "<one paragraph explaining the classification and why>",
  "customer_response": "<draft message to send the customer, professional and direct>",
  "scope_doc_update": "<one sentence: what to add to the scope doc change log>"
}}

Classification rules:
- clarification: same goal, better definition, usually reduces or maintains scope
- expansion: same goal, additional features or coverage, adds work
- pivot: fundamentally different goal, requires restarting scoping

For customer_response:
- Do not commit to anything until impact is assessed (even if it is clarification, say you are updating the scope doc)
- For expansion and pivot: be honest about the tradeoff, do not hedge with "might" and "maybe"
- Keep it under 100 words, professional but direct
- Do not start with "I hope this message finds you well" or similar filler

Return only the JSON object, no other text."""


def classify_scope_change(
    context: str,
    original_scope: str,
    change_request: str,
) -> ScopeChangeAssessment:
    """Classify a scope change and generate a customer response."""
    prompt = CLASSIFY_PROMPT.format(
        context=context,
        original_scope=original_scope,
        change_request=change_request,
    )

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )

    raw = response.content[0].text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(lines[1:-1])

    data = json.loads(raw)

    return ScopeChangeAssessment(
        change_type=ChangeType(data["change_type"]),
        effort_delta_days=int(data["effort_delta_days"]),
        timeline_risk=data["timeline_risk"],
        rationale=data["rationale"],
        customer_response=data["customer_response"],
        scope_doc_update=data["scope_doc_update"],
    )


RISK_LABELS = {
    "low": "[LOW]",
    "medium": "[MEDIUM]",
    "high": "[HIGH]",
}

CHANGE_LABELS = {
    ChangeType.CLARIFICATION: "CLARIFICATION (accept + update doc)",
    ChangeType.EXPANSION: "EXPANSION (negotiate or decline)",
    ChangeType.PIVOT: "PIVOT (restart scoping)",
}


def print_assessment(assessment: ScopeChangeAssessment) -> None:
    """Print a formatted assessment report."""
    print("\n" + "=" * 60)
    print("SCOPE CHANGE ASSESSMENT")
    print("=" * 60)

    print(f"\nType:          {CHANGE_LABELS[assessment.change_type]}")
    print(f"Effort delta:  +{assessment.effort_delta_days} engineering days")
    print(f"Timeline risk: {RISK_LABELS[assessment.timeline_risk]}")

    print("\n--- RATIONALE ---")
    print(assessment.rationale)

    print("\n--- DRAFT CUSTOMER RESPONSE ---")
    print(assessment.customer_response)

    print("\n--- SCOPE DOC UPDATE ---")
    print(assessment.scope_doc_update)

    print("\n" + "=" * 60)
    if assessment.change_type == ChangeType.PIVOT:
        print("ACTION REQUIRED: Stop current work. Schedule a scoping call.")
    elif assessment.change_type == ChangeType.EXPANSION:
        print("ACTION REQUIRED: Send response. Wait for customer decision before proceeding.")
    else:
        print("ACTION REQUIRED: Update scope doc. Confirm with customer in writing.")
    print("=" * 60 + "\n")


DEMO_SCENARIOS = [
    {
        "name": "Scenario 1: Expansion",
        "context": (
            "6-week engagement to build a support ticket classifier for a SaaS company. "
            "Week 3 of 6. Classifier covers Tier 1 support (password resets, account access, "
            "basic how-to questions). Eval set built, first model in staging."
        ),
        "original_scope": (
            "Build and deploy a text classifier that routes Tier 1 support tickets to the correct "
            "support queue. Includes: data labeling guide, model training, eval set (200 labeled "
            "examples), API endpoint, basic admin dashboard. "
            "NOT IN SCOPE: billing questions, Tier 2/3 tickets, multi-language support."
        ),
        "change_request": (
            "Hey, while you are at it, could you also make it handle billing questions? "
            "Customers always get confused between account access and billing, so it would be "
            "great to have both. That is basically the same data, right?"
        ),
    },
    {
        "name": "Scenario 2: Clarification",
        "context": (
            "4-week engagement to build a document summarization tool for a legal firm. "
            "Week 1. Discovery just completed."
        ),
        "original_scope": (
            "Build a tool that summarizes legal documents under 50 pages and returns key clauses, "
            "dates, and parties. Integrates with their Google Drive. "
            "NOT IN SCOPE: contracts over 50 pages, court filings, multi-language documents."
        ),
        "change_request": (
            "We realized we should clarify: when we said 'legal documents' we meant specifically "
            "contract amendments and NDAs. We do not need it to handle lease agreements or "
            "employment contracts. Those go to a different team."
        ),
    },
    {
        "name": "Scenario 3: Pivot",
        "context": (
            "5-week engagement to build an internal knowledge base search tool. "
            "Week 2. Architecture designed, data pipeline in progress."
        ),
        "original_scope": (
            "Build a semantic search system over the company's internal wiki (3,000 documents). "
            "Includes: ingestion pipeline, vector store, search API, basic UI. "
            "NOT IN SCOPE: real-time document sync, user access controls, analytics dashboard."
        ),
        "change_request": (
            "Our CEO just came back from a conference and wants us to pivot. Instead of the search "
            "tool, can you build a full customer-facing chatbot that answers questions about our "
            "product? He wants it launched in 3 weeks."
        ),
    },
]


def main() -> None:
    parser = argparse.ArgumentParser(
        description="ScopeChangeEvaluator: classify scope changes and draft customer responses"
    )
    parser.add_argument(
        "--demo",
        action="store_true",
        help="Run on 3 built-in demo scenarios",
    )
    parser.add_argument("--context", help="Project context (what is being built, what week)")
    parser.add_argument(
        "--scope",
        help="Original scope statement including NOT IN SCOPE section",
    )
    parser.add_argument("--change", help="The scope change request text")
    args = parser.parse_args()

    if args.demo:
        for scenario in DEMO_SCENARIOS:
            print(f"\n{'#' * 60}")
            print(f"  {scenario['name']}")
            print(f"{'#' * 60}")
            assessment = classify_scope_change(
                context=scenario["context"],
                original_scope=scenario["original_scope"],
                change_request=scenario["change_request"],
            )
            print_assessment(assessment)
    elif args.context and args.scope and args.change:
        assessment = classify_scope_change(
            context=args.context,
            original_scope=args.scope,
            change_request=args.change,
        )
        print_assessment(assessment)
    else:
        print("Usage:")
        print("  python main.py --demo")
        print("  python main.py --context '...' --scope '...' --change '...'")
        sys.exit(1)

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. How should you classify this request?**

- A. Clarification, because the customer is giving you more information about their documents
- B. Expansion, because multi-language support adds significant new work not covered in the original scope
- C. Pivot, because processing Spanish documents requires a completely different system
- D. Clarification, because you should have asked about language requirements during discovery

**2. What is the correct immediate response?**

- A. Say yes immediately to keep the customer happy and figure out the timeline later
- B. Say no immediately to protect the delivery date
- C. Tell the customer you will assess the impact and get back to them by a specific time within 24 hours
- D. Ask the customer to submit a formal change request form before you can respond

**3. What classification should the tool return, and what is the correct action?**

- A. Expansion, because the customer is changing the agreed scope
- B. Pivot, because the goal has fundamentally changed
- C. Clarification, because the goal is the same and the scope is narrowing, not growing
- D. Expansion, because you will now need to add filtering logic to exclude Engineering docs

**4. How should you classify and respond to this request?**

- A. Expansion, because you will need to build both systems in parallel
- B. Clarification, because the goal (automated classification) is the same
- C. Pivot, because switching from an LLM approach to a rules-based system requires restarting the design and implementation
- D. Clarification, because the customer is just specifying a technical preference

**5. What is the most effective way to address this concern while preserving the customer relationship?**

- A. Agree to work verbally and take your own private notes to protect yourself
- B. Insist on the formal process and risk the relationship if they disagree
- C. Explain that the written process protects them too, by giving them a clear record of what was agreed and preventing misalignment between their own stakeholders
- D. Use the formal process anyway without telling them and send written summaries after every verbal discussion

**6. What is the correct response?**

- A. Agree to work weekends to maintain the customer relationship
- B. Decline working weekends and present the three options: accept the expansion with a timeline extension, decline the expansion, or reduce another deliverable to make room
- C. Decline working weekends and drop the expansion without discussing alternatives
- D. Ask your manager to handle this conversation

**7. Which information belongs in the change log entry?**

- A. Only the date and a brief description of the new feature
- B. The date, the original request text, the classification, the decision (approved with 4-day extension), and who approved it
- C. The date, the new delivery date, and your internal effort estimate
- D. A summary written from memory at the end of the project

**8. What should you do before assessing the third request?**

- A. Assess and approve it as you did the previous two, since the process is working
- B. Refuse the third request automatically since the project is already late
- C. Escalate to your manager or account lead, since multiple expansions indicate a scoping problem that the change management process alone cannot fix
- D. Accept the request but do not extend the timeline this time

_Answers are in `checks.json` in the lesson directory._